# Bash para Administración de Sistemas y Automatización

Bash es la herramienta principal para la gestión de servidores Linux. Este notebook se centra en patrones avanzados, manejo de archivos, monitorización de recursos y automatización de tareas administrativas.

## 1. Verificación de Archivos y Directorios

En scripting de sistemas, es fundamental verificar la existencia y permisos de archivos antes de operar con ellos.

In [ ]:
ARCHIVO="/etc/passwd"

if [[ -f "$ARCHIVO" ]]; then
    echo "Es un archivo regular"
fi

if [[ -d "/var/log" ]]; then
    echo "El directorio de logs existe"
fi

if [[ -r "$ARCHIVO" && -w "$ARCHIVO" ]]; then
    echo "Tengo permisos de lectura y escritura"
else
    echo "Permisos insuficientes"
fi

# Operadores comunes:
# -e (existe), -s (no vacío), -x (ejecutable), -L (enlace simbólico)

## 2. Automatización con Bucles y Comandos de Sistema

Patrones comunes para procesar logs o realizar backups.

In [ ]:
# Iterar sobre archivos en un directorio
for log en /var/log/*.log; do
    echo "Procesando: $(basename "$log")"
done

# Leer un archivo línea por línea (Patrón seguro)
while IFS= read -r linea; do
    echo "Contenido: $linea"
done < "archivo.txt"

## 3. Manejo de Errores y Seguridad en Scripts

Para scripts robustos, se recomienda usar `set` para controlar el comportamiento ante fallos.

In [ ]:
set -e          # Detener el script si un comando falla
set -u          # Error si se usa una variable no definida
set -o pipefail # Capturar errores en tuberías (pipes)

# Función de limpieza (Trap)
# Ejecuta código al salir, ideal para borrar archivos temporales
cleanup() {
    echo "Limpiando temporales..."
    rm -f /tmp/script_tmp_*
}
trap cleanup EXIT

## 4. Monitorización de Recursos

Uso de comandos de sistema para alertar sobre el estado del servidor.

In [ ]:
# Uso de disco
UMBRAL=80
uso=$(df / | grep / | awk '{ print $5 }' | sed 's/%//')

if [[ $uso -gt $UMBRAL ]]; then
    echo "ALERTA: Espacio en disco crítico: $uso%"
fi

# Carga de CPU
carga=$(uptime | awk -F'load average:' '{ print $2 }' | cut -d, -f1)
echo "Carga actual: $carga"

## 5. Herramientas de Procesamiento de Texto (AWK, SED, GREP)

Son esenciales para manipular datos de configuración y logs.

In [ ]:
# GREP: Buscar patrones
grep -i "error" /var/log/syslog

# SED: Reemplazo de texto (Stream Editor)
# Cambia 'localhost' por '127.0.0.1' en un archivo
sed -i 's/localhost/127.0.0.1/g' config.conf

# AWK: Procesamiento por columnas
# Listar usuarios y sus shells del archivo passwd
awk -F: '{ print $1 " usa " $7 }' /etc/passwd

## 6. Red y Servicios

Gestión de conectividad y estado de demonios.

In [ ]:
# Verificar si un puerto está abierto
(nc -zv 127.0.0.1 80) &>/dev/null && echo "Puerto 80 OK" || echo "Puerto 80 CERRADO"

# Verificar estado de un servicio (Systemd)
if systemctl is-active --quiet nginx; then
    echo "Nginx está corriendo"
else
    echo "Nginx está caído, reiniciando..."
    # systemctl restart nginx
fi

## 7. Automatización de Tareas (Cron)

Aunque no es sintaxis Bash, es donde viven los scripts de automatización.

```bash
# Editar crontab: crontab -e
# Formato: min hora dia mes dia_semana comando
0 2 * * * /home/usuario/scripts/backup.sh  # Ejecuta a las 2 AM diario
```